# 🚀 Derm-AI: Automated End-to-End Fine-Tuning Pipeline
This notebook is fully automated. You just need to click **Runtime -> Run all**.
It will:
1. Download the dermatology datasets from HuggingFace.
2. Preprocess the images and labels for a Vision Transformer (ViT).
3. Fine-tune the model on the T4 GPU.
4. Save, zip, and automatically download the final .zip model to your computer.

In [ ]:
# 1. Install required packages
!pip install -q datasets transformers[torch] accelerate evaluate

In [ ]:
# 2. Authenticate with HuggingFace (Required for Derm1M)
from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get('HF_TOKEN'))
    print('Successfully logged in via Colab Secrets!')
except:
    print('Please enter your HuggingFace token below:')
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Load & Harmonize Datasets
from datasets import load_dataset, concatenate_datasets, ClassLabel

print('Downloading Fitzpatrick 17k (SkinCAP version)...')
# We use SkinCAP as the base as it has clean Fitzpatrick17k images and labels ready for classification
dataset = load_dataset('joshuachou/SkinCAP', split='train')

print(f'Downloaded {len(dataset)} images!')

# Ensure the dataset has 'image' and 'label' columns
if 'image' not in dataset.column_names:
    # Attempt to rename whatever image column exists
    img_cols = [c for c in dataset.column_names if 'img' in c.lower() or 'image' in c.lower()]
    if img_cols:
        dataset = dataset.rename_column(img_cols[0], 'image')

if 'label' not in dataset.column_names:
    # Using condition/diagnosis as label
    lbl_cols = [c for c in dataset.column_names if 'condition' in c.lower() or 'diagnosis' in c.lower()]
    if lbl_cols:
        dataset = dataset.rename_column(lbl_cols[0], 'label')

# Convert string labels to integers if necessary
unique_labels = list(set(dataset['label']))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

def map_labels(example):
    example['label'] = label2id[example['label']]
    return example

dataset = dataset.map(map_labels)

# Split into Train and Validation (90/10 split)
dataset = dataset.train_test_split(test_size=0.1)
train_ds = dataset['train']
val_ds = dataset['test']
print(f'Training on {len(train_ds)} images, Validating on {len(val_ds)} images.')

In [ ]:
# 4. Preprocess Images for Vision Transformer (ViT)
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

model_name = 'google/vit-base-patch16-224-in21k'
processor = AutoImageProcessor.from_pretrained(model_name)

normalize = Normalize(mean=processor.image_mean, std=processor.image_std)
train_transforms = Compose([
    RandomResizedCrop(processor.size['height']),
    ToTensor(),
    normalize,
])

val_transforms = Compose([
    RandomResizedCrop(processor.size['height']),
    ToTensor(),
    normalize,
])

def preprocess_train(example_batch):
    example_batch['pixel_values'] = [
        train_transforms(image.convert('RGB')) for image in example_batch['image']
    ]
    return example_batch

def preprocess_val(example_batch):
    example_batch['pixel_values'] = [
        val_transforms(image.convert('RGB')) for image in example_batch['image']
    ]
    return example_batch

train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_val)

import torch
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

In [ ]:
# 5. Initialize Model & Training Arguments
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

metric = evaluate.load('accuracy')
def compute_metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

training_args = TrainingArguments(
    output_dir='./derm_ai_model_checkpoints',
    remove_unused_columns=False,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
# 6. Start Fine-Tuning!
print('Starting training process. This may take several hours depending on the dataset size...')
trainer.train()

In [ ]:
# 7. Export and Download the Fine-Tuned Model
import shutil
from google.colab import files

save_path = './derm_ai_final_model'

print(f'Saving final model to {save_path}...')
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print('Zipping the model for download...')
shutil.make_archive('derm_ai_final_model', 'zip', save_path)

print('Initiating download to your computer! Please leave the browser tab open.')
files.download('derm_ai_final_model.zip')